# VisClick — Part C (step 1): pull repo + start data acquisition

**You already did:** GitHub repo, Google Drive folder `visclick`, Colab GPU.

**This notebook (only this for now):** always **pull the latest code** from GitHub, then download the **first** datasets (CLAY + UI-Vision). Other datasets (Zenodo, RICO, VINS) come in a **later** step — see `VisClick_Detailed_Plan.md` Part C.2.

**Drive path in Colab:** `My Drive/visclick` → `/content/drive/MyDrive/visclick`


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 1) Get latest code: clone if first time, else pull
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
ROOT = "/content/visclick"
if not os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "clone", REPO, ROOT], check=True)
    print("Cloned to", ROOT)
else:
    subprocess.run(["git", "-C", ROOT, "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", ROOT, "pull", "--rebase", "origin", "main"], check=False)
    print("Pulled latest in", ROOT)


In [ ]:
# 2) Work inside the repo (configs, scripts, reports paths)
%cd /content/visclick


In [ ]:
# 3) Drive root (must match your Google Drive folder name)
import os
DRIVE = "/content/drive/MyDrive/visclick"
for sub in (
    "data/raw", "data/clay", "data/vins", "data/webui", "data/ui_vision",
    "data/unified", "data/source_train", "data/desktop_labeled", "data/desktop_test",
    "weights/baseline_source", "weights/experiments", "videos",
):
    os.makedirs(os.path.join(DRIVE, sub), exist_ok=True)
print("DRIVE =", DRIVE)


In [ ]:
# 4) Tools for downloads
!pip install -q "huggingface_hub[cli]>=0.20" tqdm


## C.2.2 — CLAY (labels; images need RICO later)

Clones into `Drive/.../data/raw/clay` and copies into `data/clay/`.


In [ ]:
import os, shutil, subprocess
DRIVE = "/content/drive/MyDrive/visclick"
raw = os.path.join(DRIVE, "data", "raw")
os.makedirs(raw, exist_ok=True)
os.chdir(raw)
if not os.path.isdir("clay"):
    subprocess.run(
        ["git", "clone", "https://github.com/google-research-datasets/clay.git", "clay"],
        check=True,
    )
clay_src = os.path.join(raw, "clay")
clay_dst = os.path.join(DRIVE, "data", "clay")
os.makedirs(clay_dst, exist_ok=True)
for name in os.listdir(clay_src):
    s, d = os.path.join(clay_src, name), os.path.join(clay_dst, name)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    else:
        shutil.copy2(s, d)
print("CLAY ->", clay_dst)


## C.2.5 — UI-Vision (desktop benchmark, ~781 MB)

Saves under `data/ui_vision/`. Needs a Hugging Face **read** token if the dataset asks for it:  
Hub → Settings → Access Tokens → add token and run `huggingface-cli login` once in Colab.


In [ ]:
import os, subprocess, sys
DRIVE = "/content/drive/MyDrive/visclick"
out = os.path.join(DRIVE, "data", "ui_vision")
os.makedirs(out, exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub"], check=False)
# If 401: run in a new cell: !huggingface-cli login
r = subprocess.run(
    ["hf", "download", "ServiceNow/ui-vision", "--repo-type", "dataset", "--local-dir", out],
    check=False,
)
print("UI-Vision ->", out, "exit", r.returncode)


## Stop here for this session

**Check:** In Google Drive, you should see new files under `visclick/data/clay` and `visclick/data/ui_vision`.

**Next step (not in this notebook yet):** Part C.2.1 RICO (often manual download), C.2.6 Zenodo unified bundle, C.2.4 VINS, then C.4 EDA and C.5 `source_train` assembly — we add the next Colab notebook only when you are ready.

**Local:** Zotero + `reports/literature_table.csv` for papers (plan C.1).
